# Quantizing Qwen to INT4 (W4A16) with `llm-compressor`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxkimambo/inference-engineering-deep-dive/blob/main/docs/notebooks/quantization-qwen-w4a16.ipynb)

Companion notebook to **[§5.1 Quantization](https://inference.kimambo.de/chapters/techniques/quantization/)**.
It runs the whole post-training quantization (PTQ) pipeline end to end — load BF16 weights,
calibrate, quantize to **W4A16** (4-bit weights, 16-bit activations) with **GPTQ**, save the
compressed checkpoint, and sanity-check a generation.

Every step prints what it is doing and how long it took, so you can watch the pipeline work.

> **⚙️ Runtime.** This defaults to **Qwen2.5-0.5B-Instruct** so it fits and runs in a few minutes on
> a **free Colab T4 (16 GB)**. The chapter walks through the 7B model; to reproduce those exact size
> numbers, set `MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"` below and switch to a **Colab Pro L4 or A100**
> runtime — 7B GPTQ will OOM a T4.
>
> Enable the GPU first: **Runtime → Change runtime type → T4 GPU**.

## 0 · Confirm the GPU

Quantization needs a GPU. This should show a Tesla **T4** (free tier) — or an L4/A100 if you
upgraded the runtime for the 7B model.

In [ ]:
!nvidia-smi

## 1 · Install `llm-compressor`

We use [`llm-compressor`](https://github.com/vllm-project/llm-compressor), the vLLM-native
quantization library — its output loads straight into vLLM/SGLang.

> **⚠️ If you hit a pydantic `ValidationError`** when building the recipe (Step 8), it's a
> `transformers` / `compressed-tensors` / `pydantic` version mismatch — upgrade the trio with
> `pip install -U llmcompressor transformers compressed-tensors`. Pinning an old `llmcompressor`
> also fixes it, but on Colab that can drag `torch` back and break the pre-installed packages, so
> prefer upgrading.

In [ ]:
# Unpinned so the install matches Colab's pre-installed torch (a pin can drag
# torch back and break torchvision/gradio). Pulls transformers, datasets,
# compressed-tensors, pydantic — a self-consistent set.
!pip install -q llmcompressor hf_transfer

## 2 · Set up logging & Hugging Face access

A tiny helper so every step announces itself and reports its wall-clock time. `llm-compressor`
also logs its own progress (via `loguru`) during calibration — you'll see both streams.

In [ ]:
import logging, time, contextlib

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-5s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("quant")

@contextlib.contextmanager
def step(msg):
    """Log a step's start, then its elapsed time on completion."""
    log.info(msg)
    t0 = time.perf_counter()
    yield
    log.info(f"done in {time.perf_counter() - t0:,.1f}s")

### Sign in to Hugging Face (faster downloads)

The BF16 weights and calibration data pull from the Hugging Face Hub. Authenticating raises your rate
limits (and unlocks any gated models), and `hf_transfer` — a Rust downloader — saturates Colab's
bandwidth for a noticeably faster pull.

Add your token once in Colab's **🔑 Secrets** panel (left sidebar) as `HF_TOKEN` with *Notebook
access* on, then run the cell below. Grab a token at
[huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

In [ ]:
import os
from huggingface_hub import login

# Read HF_TOKEN from Colab's Secrets panel (the 🔑 in the left sidebar);
# fall back to an environment variable when running outside Colab.
hf_token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN") or hf_token
except Exception:
    pass

# Enable the faster Rust transfer backend if it's installed.
try:
    import hf_transfer  # noqa: F401
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
except Exception:
    pass

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token)          # authenticated pulls: higher rate limits + gated models
    log.info("Hugging Face: authenticated, hf_transfer enabled")
else:
    log.warning("Hugging Face: no HF_TOKEN found (anonymous). Add it in the 🔑 Secrets panel.")

## 3 · Pick the model and recipe knobs

**W4A16** = 4-bit weights, 16-bit activations. The activations stay at 16-bit on purpose: the
[sensitivity ladder](https://inference.kimambo.de/chapters/techniques/quantization/#what-the-sensitivity-ladder)
says weights tolerate quantization best, so we crush them and leave everything else alone.

- `GROUP_SIZE = 128` — weights share one scale factor per group of 128 (finer = better quality,
  more scales to store).
- `NUM_CALIBRATION_SAMPLES` — how many real samples GPTQ sees to measure value ranges. 256–512 is
  plenty; more is slower.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"   # swap to "Qwen/Qwen2.5-7B-Instruct" on an L4/A100
GROUP_SIZE = 128
NUM_CALIBRATION_SAMPLES = 256
MAX_SEQ_LEN = 2048

log.info(f"Model: {MODEL_ID}")
log.info(f"Recipe: W4A16, group_size={GROUP_SIZE}, calib_samples={NUM_CALIBRATION_SAMPLES}")

## 4 · Load the BF16 model

`dtype="auto"` loads the weights in their native precision (BF16 for Qwen). We also print the
parameter count and the in-memory weight size — this is the "before" number the quantization
will shrink.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

with step(f"Loading {MODEL_ID} in native precision"):
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype="auto")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

n_params = sum(p.numel() for p in model.parameters())
bytes_per = next(model.parameters()).element_size()
native_gb = n_params * bytes_per / 1e9

log.info(f"Parameters   : {n_params/1e9:.3f} B")
log.info(f"Native dtype : {next(model.parameters()).dtype} ({bytes_per} bytes/param)")
log.info(f"Weight size  : ≈ {native_gb:.2f} GB (the 'before' number)")

## 5 · Inspect the model — layers and what they mean

Before touching the recipe, look at what you loaded. The module tree *is* the map you target:
`ignore=["lm_head"]` and `re:.*self_attn.*` refer to exactly these paths. `print(model)` dumps the
whole tree.

In [ ]:
print(model)

Reading the tree top to bottom (Qwen, like most decoder-only LLMs):

| Module path | What it is | Sensitivity |
|---|---|---|
| `model.embed_tokens` | token → vector lookup table | sensitive (leave in BF16) |
| `model.layers.{i}.self_attn.{q,k,v,o}_proj` | attention projections | the `k/v_proj` feed the KV cache |
| `model.layers.{i}.mlp.{gate,up,down}_proj` | the FFN — biggest, most robust | quantize hardest; `down_proj` least |
| `model.layers.{i}.*_layernorm` | RMSNorm — tiny | never quantized |
| `model.norm` | final norm | tiny |
| `lm_head` | vector → vocab logits (output head) | sensitive (leave in BF16) |

The `config` gives the shape numbers behind that tree:

In [ ]:
cfg = model.config
print("architecture      :", cfg.architectures[0])
print("hidden layers     :", cfg.num_hidden_layers)      # → the layers.{0..N-1} range for edge-layer regexes
print("hidden size       :", cfg.hidden_size)
print("intermediate size :", cfg.intermediate_size)      # the FFN width (gate/up/down_proj)
print("attention heads   :", cfg.num_attention_heads)
print("KV heads          :", cfg.num_key_value_heads)    # < attention heads ⇒ grouped-query attention (GQA)
print("vocab size        :", cfg.vocab_size)

**Enumerate the Linear layers** — this is the recon you do before writing a recipe. Every name
printed here is a valid `targets`/`ignore` path; collapsing the layer index (`.0.` → `.N.`)
shows the handful of *families* you actually target.

In [ ]:
import re
import torch.nn as nn
from collections import Counter

linears = [(n, tuple(m.weight.shape)) for n, m in model.named_modules() if isinstance(m, nn.Linear)]
families = Counter(re.sub(r"\.\d+\.", ".N.", n) for n, _ in linears)

print(f"{len(linears)} Linear layers, in {len(families)} families:\n")
for name, count in families.items():
    example = next(shape for n, shape in linears if re.sub(r"\.\d+\.", ".N.", n) == name)
    print(f"{count:4d}  {name:34s} weight{example}")

**Where do the parameters actually live?** Quantization only pays off where the weight bytes
are. In a small model the embedding + head can be a surprising share; in a 7B the transformer
layers dominate — which is why weights-only W4A16 gets most of the win.

In [ ]:
from collections import defaultdict

buckets = defaultdict(int)
for name, p in model.named_parameters():
    if ".layers." in name:   key = f"transformer layers (×{cfg.num_hidden_layers})"
    elif "embed" in name:    key = "embed_tokens"
    elif name.startswith("lm_head"): key = "lm_head"
    else:                    key = "norms / other"
    buckets[key] += p.numel()

total = sum(buckets.values())
for key, n in sorted(buckets.items(), key=lambda kv: -kv[1]):
    print(f"{key:28s} {n/1e6:8.1f} M  ({100*n/total:4.1f}%)")

**Inspect a single weight tensor.** The gap between the *average* magnitude and the *absmax*
is the outlier problem quantization fights — one big value stretches the scale for the whole
group (the next step quantizes a real group by hand so you can watch it happen).

In [ ]:
w = model.model.layers[0].mlp.down_proj.weight
print("path   : model.layers.0.mlp.down_proj.weight")
print("shape  :", tuple(w.shape))
print("dtype  :", w.dtype)
print("device :", w.device)
print(f"min/max: {w.min().item():+.4f} / {w.max().item():+.4f}")
print(f"mean|w|: {w.abs().mean().item():.4f}")
print(f"absmax : {w.abs().max().item():.4f}   <- one outlier sets the whole group's scale")

## 6 · Peek under the hood: quantize 8 real weights by hand

Before running the real algorithm, let's reproduce the chapter's
[Step 1 math](https://inference.kimambo.de/chapters/techniques/quantization/#step-1-the-math-on-eight-real-weights)
on an **actual group of 8 weights** pulled from the model. This is plain round-to-nearest (RTN) —
the simplest possible scheme — so you can *see* the scale factor, the rounding, and the error
before GPTQ does it more cleverly across billions of weights.

In [ ]:
# One real group of 8 weights from the first layer's MLP down_proj.
group = model.model.layers[0].mlp.down_proj.weight.data[0, :8].float()

qmax = 7                                    # signed INT4 codes run [-7, 7]
absmax = group.abs().max()                  # the outlier sets the scale
scale = absmax / qmax                       # size of one quantization step
q = torch.clamp(torch.round(group / scale), -qmax, qmax)   # the stored INT4 codes
dequant = q * scale                         # what you get back
error = dequant - group

torch.set_printoptions(precision=4, sci_mode=False)
print("weights (BF16) :", group)
print("absmax         :", round(absmax.item(), 5))
print("scale S        :", round(scale.item(), 5))
print("quantized q    :", q.to(torch.int8).tolist())
print("dequantized w' :", dequant)
print("error (w'-w)   :", error)
print("max abs error  :", round(error.abs().max().item(), 5))

Notice the pattern from the chapter: the value that *set* the scale comes back exactly, while
small values near a big outlier round hardest — sometimes all the way to `0`. That's why **group
size matters**: a smaller group means an outlier inflates the scale for fewer neighbours. GPTQ
(next) improves on this by using loss-curvature information to compensate the not-yet-quantized
weights for each rounding error, minimizing the layer's *output* error rather than each weight's.

## 7 · Prepare calibration data

GPTQ needs a few hundred representative samples to measure real activation/weight ranges. We use
`ultrachat_200k` (general chat). **Use domain-matched data** — for a code model, calibrate on
code, not chat. We render each conversation through the chat template, then tokenize.

In [ ]:
from datasets import load_dataset

with step("Loading + preparing calibration data"):
    ds = load_dataset("HuggingFaceH4/ultrachat_200k", split=f"train_sft[:{NUM_CALIBRATION_SAMPLES}]")
    ds = ds.shuffle(seed=42)

    def to_text(example):
        return {"text": tokenizer.apply_chat_template(example["messages"], tokenize=False)}

    def tokenize(example):
        return tokenizer(example["text"], max_length=MAX_SEQ_LEN, truncation=True, add_special_tokens=False)

    ds = ds.map(to_text)
    ds = ds.map(tokenize, remove_columns=ds.column_names)

log.info(f"Calibration samples ready: {len(ds)}")

## 8 · Define the GPTQ recipe

`GPTQModifier` is the smart PTQ algorithm — **not** round-to-nearest.

- `targets="Linear"` — every `nn.Linear` is a candidate to quantize.
- `scheme="W4A16"` — 4-bit weights, 16-bit activations, `group_size=128`.
- `ignore=["lm_head"]` — keep the output head in BF16 (it's sensitive; sensitivity ladder).

This is the one control that matters most in practice. See **Targeting different layers** near the
end for how to protect edge layers, skip attention, or spare `down_proj`.

In [ ]:
from llmcompressor.modifiers.gptq import GPTQModifier

recipe = GPTQModifier(
    targets="Linear",       # quantize every nn.Linear…
    scheme="W4A16",         # …to 4-bit weights / 16-bit activations, group_size 128
    ignore=["lm_head"],     # …except the output head (kept in BF16)
)
log.info(f"Recipe: GPTQ {recipe.scheme}, targets={recipe.targets}, ignore={recipe.ignore}")

## 9 · Calibrate + quantize (the actual work)

`oneshot()` runs the calibration data through the model, builds the GPTQ Hessians, and quantizes
each targeted layer **in place**. This is the slow step — a minute or two for 0.5B on a T4, ~20–60
min for 7B on an L4. Watch `llm-compressor`'s per-layer progress logs.

In [ ]:
from llmcompressor import oneshot

with step("Calibrating + quantizing (GPTQ one-shot)"):
    oneshot(
        model=model,
        dataset=ds,
        recipe=recipe,
        max_seq_length=MAX_SEQ_LEN,
        num_calibration_samples=NUM_CALIBRATION_SAMPLES,
    )

log.info("Model is now quantized in place.")

## 10 · Save the compressed checkpoint and measure the payoff

`save_compressed=True` writes the packed 4-bit weights. We then measure the on-disk size and
compare it to the BF16 "before" number from Step 4.

In [ ]:
import os

SAVE_DIR = MODEL_ID.split("/")[-1] + f"-W4A16-G{GROUP_SIZE}"

with step(f"Saving compressed checkpoint → {SAVE_DIR}"):
    model.save_pretrained(SAVE_DIR, save_compressed=True)
    tokenizer.save_pretrained(SAVE_DIR)

def dir_size_gb(path):
    total = sum(
        os.path.getsize(os.path.join(root, f))
        for root, _, files in os.walk(path) for f in files
    )
    return total / 1e9

quant_gb = dir_size_gb(SAVE_DIR)
print(f"BF16 weights (in memory) : {native_gb:6.2f} GB")
print(f"W4A16 checkpoint (disk)  : {quant_gb:6.2f} GB")
print(f"Shrink factor            : {native_gb / quant_gb:6.2f}x")

## 11 · Sanity-check a generation

The quantized model still generates. This is a smoke test, **not** a quality measurement — for
that, run perplexity / MMLU / your custom eval against the original BF16 weights (chapter §5.1.3).

In [ ]:
with step("Generating a sample completion from the quantized model"):
    messages = [{"role": "user", "content": "In one sentence, what does INT4 quantization trade away?"}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(inputs, max_new_tokens=120, do_sample=False)

print(tokenizer.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

## 12 · Evaluate for regressions — the pre-release gate

The sanity generation above proves the model *runs*. It does **not** prove the quantization was
free. This section is the check you'd actually run before shipping: measure quality **against the
original BF16 weights** and decide, on evidence, whether the 4-bit model is good enough to serve.

Four principles make this a real gate rather than theatre:

1. **Baseline-relative, not absolute.** A 45% score means nothing alone. `BF16 45.2% → W4A16 44.9%`
   is the signal. Every metric is a *delta* against the unquantized model, run through the *same*
   harness with the *same* prompts and sampling params.
2. **Beat the noise floor.** LLMs are non-deterministic, so scores wobble run to run. We use greedy
   decoding (`do_sample=False`) and fixed data to shrink that, and treat any change smaller than the
   run-to-run spread as *no change*. A regression only counts if it clears the noise band.
3. **Cheap tripwire, expensive gate.** Perplexity is fast and catches gross breakage; benchmarks
   catch capability loss; but your **custom eval** — your product's real prompts — is what actually
   gates the deploy.
4. **Most-aggressive-that-passes.** Pick the smallest, fastest quant setting that still clears the
   gate, and no further. If it regresses, dial back one notch (add sensitive layers to `ignore`, or
   W4A16 → W8A16) and re-measure.

We evaluate each model by **loading it from disk, scoring, then freeing it** — so this scales from
0.5B on a T4 up to 7B+, where the two models can't co-reside in VRAM.

In [ ]:
import gc, math, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# The quantized model is still in VRAM from Step 9 — free it; every eval below
# reloads from disk so baseline and candidate never fight for memory.
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

BASELINE = MODEL_ID          # original BF16 weights
CANDIDATE = SAVE_DIR         # the W4A16 checkpoint we just wrote

### 12a · Perplexity — the cheap tripwire

Perplexity is how *surprised* the model is by real text (lower = better). It's a fast first check:
run identical held-out text (WikiText-2) through both models and compare. A small increase is
expected and fine; a large jump means something broke. It's a tripwire, **not** the gate — a model
can hold perplexity and still lose reasoning ability.

In [ ]:
from datasets import load_dataset

@torch.no_grad()
def perplexity(model, tokenizer, max_length=2048, stride=1024):
    """Sliding-window perplexity over WikiText-2 test (the standard HF recipe)."""
    data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    enc = tokenizer("\n\n".join(data["text"]), return_tensors="pt")
    seq_len = enc.input_ids.size(1)
    nll_sum, n_tokens, prev = 0.0, 0, 0
    for begin in range(0, seq_len, stride):
        end = min(begin + max_length, seq_len)
        trg_len = end - prev                       # tokens scored in this window
        ids = enc.input_ids[:, begin:end].to(model.device)
        target = ids.clone()
        target[:, :-trg_len] = -100                # only score the new tokens
        loss = model(ids, labels=target).loss      # mean NLL over scored tokens
        nll_sum += loss.item() * trg_len
        n_tokens += trg_len
        prev = end
        if end == seq_len:
            break
    return math.exp(nll_sum / n_tokens)

def ppl_for(path):
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForCausalLM.from_pretrained(path, dtype="auto", device_map="cuda").eval()
    val = perplexity(m, tok)
    del m; gc.collect(); torch.cuda.empty_cache()
    return val

with step("Perplexity: BF16 baseline"):
    ppl_bf16 = ppl_for(BASELINE)
with step("Perplexity: W4A16 candidate"):
    ppl_w4a16 = ppl_for(CANDIDATE)

ppl_delta_pct = 100 * (ppl_w4a16 - ppl_bf16) / ppl_bf16
print(f"\nwikitext PPL   BF16={ppl_bf16:.3f}   W4A16={ppl_w4a16:.3f}   Δ={ppl_delta_pct:+.2f}%")

### 12b · Capability benchmarks — did it get dumber?

Perplexity misses capability loss. Standard academic suites — knowledge, reasoning, common sense —
catch it. We use [EleutherAI's `lm-evaluation-harness`](https://github.com/EleutherAI/lm-evaluation-harness),
the de-facto standard, and run the **same tasks on both models**.

The cell below runs a **fast, small** subset so it finishes on a free T4. The commented block is the
**production** version: the full suite, no sample cap — what you'd actually run on a release
candidate (hours on a real GPU, not a T4).

In [ ]:
!pip install -q lm-eval

In [ ]:
from lm_eval import simple_evaluate

# --- fast Colab demo: small tasks, capped samples ---
TASKS = ["arc_easy", "hellaswag"]     # measurable on a 0.5B model in minutes
LIMIT = 100                            # samples/task; the noise band widens as this shrinks

# --- production gate (uncomment; expect hours on a real GPU) ---
# TASKS = ["mmlu", "arc_challenge", "gsm8k", "hellaswag", "truthfulqa_mc2", "winogrande"]
# LIMIT = None                         # full test sets

def headline(task_result):
    """First real accuracy metric for a task (skip stderr / alias)."""
    for k, v in task_result.items():
        if isinstance(v, (int, float)) and "stderr" not in k and k != "alias":
            return k, v
    return "n/a", float("nan")

def bench(path):
    out = simple_evaluate(model="hf", model_args=f"pretrained={path},dtype=auto",
                          tasks=TASKS, limit=LIMIT, batch_size="auto")
    gc.collect(); torch.cuda.empty_cache()
    return {t: headline(out["results"][t]) for t in TASKS}

with step("Benchmarks: BF16 baseline"):
    bench_bf16 = bench(BASELINE)
with step("Benchmarks: W4A16 candidate"):
    bench_w4a16 = bench(CANDIDATE)

print()
for t in TASKS:
    (metric, a), (_, b) = bench_bf16[t], bench_w4a16[t]
    print(f"{t:14s} {metric:20s} BF16={a:.4f}  W4A16={b:.4f}  Δ={b-a:+.4f}")

### 12c · Custom eval — the actual gate

Benchmarks tell you the model is *broadly* fine; they don't tell you it still does **your** job.
The custom eval is a golden set of prompts from your real workload, each with an automatic grader
(exact match, regex, JSON-schema validation, or an LLM judge). This is what gates the deploy — a
model can hold MMLU and still regress on your domain.

The pattern below uses a tiny exact-match set. In production this is hundreds of domain prompts with
real graders, versioned alongside the model.

In [ ]:
# Replace with your real workload prompts + graders.
GOLDEN = [
    {"q": "What is the capital of France? Answer in one word.", "a": "paris"},
    {"q": "What is 17 * 3? Answer with just the number.",       "a": "51"},
    {"q": "Which is larger, 9.9 or 9.11? Answer with the number.", "a": "9.9"},
    {"q": "Complete: The opposite of 'hot' is ___.",            "a": "cold"},
]

@torch.no_grad()
def custom_pass_rate(path):
    tok = AutoTokenizer.from_pretrained(path)
    m = AutoModelForCausalLM.from_pretrained(path, dtype="auto", device_map="cuda").eval()
    hits = 0
    for ex in GOLDEN:
        ids = tok.apply_chat_template([{"role": "user", "content": ex["q"]}],
                                      add_generation_prompt=True, return_tensors="pt").to(m.device)
        out = m.generate(ids, max_new_tokens=32, do_sample=False)        # greedy = reproducible
        text = tok.decode(out[0][ids.shape[-1]:], skip_special_tokens=True)
        hits += ex["a"].lower() in text.lower()
    del m; gc.collect(); torch.cuda.empty_cache()
    return hits / len(GOLDEN)

with step("Custom eval: BF16 baseline"):
    custom_bf16 = custom_pass_rate(BASELINE)
with step("Custom eval: W4A16 candidate"):
    custom_w4a16 = custom_pass_rate(CANDIDATE)

print(f"\ncustom pass-rate   BF16={custom_bf16:.2%}   W4A16={custom_w4a16:.2%}")

### 12d · The release decision

Put every metric in one table, apply your **pre-agreed thresholds**, and let the numbers decide. The
thresholds here are illustrative — set yours from the noise band you measure by running the baseline
against *itself* twice.

In [ ]:
# Pre-agreed gates (tune to your measured noise floor).
MAX_PPL_INCREASE_PCT = 2.0     # perplexity may rise at most 2%
MAX_BENCH_DROP = 0.01          # each benchmark may drop at most 1 point
MIN_CUSTOM_PASS = 0.95         # custom eval must hold >= 95% of baseline

rows, verdicts = [], []

d = ppl_delta_pct
ok = d <= MAX_PPL_INCREASE_PCT
rows.append(("perplexity (wikitext)", f"{ppl_bf16:.3f}", f"{ppl_w4a16:.3f}", f"{d:+.2f}%", "PASS" if ok else "FAIL"))
verdicts.append(ok)

for t in TASKS:
    (_, a), (_, b) = bench_bf16[t], bench_w4a16[t]
    ok = (a - b) <= MAX_BENCH_DROP
    rows.append((t, f"{a:.4f}", f"{b:.4f}", f"{b-a:+.4f}", "PASS" if ok else "FAIL"))
    verdicts.append(ok)

ok = custom_w4a16 >= MIN_CUSTOM_PASS * custom_bf16
rows.append(("custom eval", f"{custom_bf16:.2%}", f"{custom_w4a16:.2%}",
             f"{(custom_w4a16-custom_bf16):+.2%}", "PASS" if ok else "FAIL"))
verdicts.append(ok)

print(f"{'metric':24s} {'BF16':>10s} {'W4A16':>10s} {'delta':>10s}   {'gate':>4s}")
print("-" * 64)
for name, a, b, dlt, v in rows:
    print(f"{name:24s} {a:>10s} {b:>10s} {dlt:>10s}   {v:>4s}")
print("-" * 64)
print("RELEASE" if all(verdicts) else "BLOCK — regression exceeds a gate; dial the recipe back and re-run")

### What a full production gate adds

The four checks above are the core. A real release process wraps them with:

- **LLM-as-judge on generative quality.** Multiple-choice benchmarks miss tone, formatting, and
  instruction-following. Generate from both models on your prompts, then have a *stronger* model
  score the pair **blind** (1–5, or A/B preference) and aggregate. Watch for position bias — swap
  the order half the time.
- **Format & safety regression.** Re-run your JSON-schema / function-calling suite and your safety
  and refusal set. Quantization can quietly break structured output or shift refusal behaviour even
  when accuracy looks fine.
- **Long-context integrity.** The KV cache and RoPE interact with low precision; test needle-in-a-
  haystack at your real context length, not just short prompts.
- **You actually got faster.** Confirm the point of the exercise: benchmark served throughput and
  p50/p99 latency on vLLM (`vllm bench throughput`) against the BF16 baseline — a quant that doesn't
  speed you up isn't worth the quality risk.
- **Statistical honesty.** Report confidence intervals (lm-eval prints stderr) and run the gate on a
  fixed eval commit so results are comparable across candidates.

## Targeting different layers

`targets` says *what to quantize*; `ignore` says *what to leave in BF16*. Both accept module
**class names**, exact **module paths**, or **regex** with a `re:` prefix. This is how you walk the
sensitivity ladder in practice. Print `model` to see Qwen's module paths. Recipes from least to
most conservative:

```python
# Default — quantize every Linear, keep only the output head
GPTQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])

# Protect the edge layers — leave first 2 + last 2 transformer blocks in BF16
GPTQModifier(targets="Linear", scheme="W4A16",
             ignore=["lm_head", "re:model\\.layers\\.(0|1|22|23)\\..*"])

# MLP-only — skip all attention projections (most sensitive component)
GPTQModifier(targets="Linear", scheme="W4A16",
             ignore=["lm_head", "re:.*self_attn.*"])

# Spare a known-sensitive projection — down_proj often carries outliers
GPTQModifier(targets="Linear", scheme="W4A16",
             ignore=["lm_head", "re:.*down_proj"])
```

The layer indices in the edge-layer regex depend on the model: Qwen2.5-0.5B has 24 layers
(`0..23`), the 7B has 28 (`0..27`). Adjust to `(0|1|<n-2>|<n-1>)`.

> **🎯 The targeting workflow.** Start with **default**, run your evals. If quality regresses, don't
> abandon 4-bit — **add the regression's likely culprits to `ignore`** and re-run: edge layers, then
> attention, then `down_proj`. You're searching for the smallest set of BF16 exceptions that recovers
> quality.

> **💾 Memory tip for big models.** Add `sequential_targets=["Qwen2DecoderLayer"]` to the modifier to
> quantize **one decoder layer at a time**, keeping only that layer's activations in memory. Essential
> when the model barely fits — it's how the same recipe scales from 7B to 70B+.

## Use AWQ instead (one import)

AWQ often edges out GPTQ at 4-bit — it scales up salient weight channels (judged by activation
magnitude) before quantizing, so rounding hurts them less. Same `oneshot(...)` call, different
modifier:

```python
from llmcompressor.modifiers.awq import AWQModifier

recipe = AWQModifier(targets="Linear", scheme="W4A16", ignore=["lm_head"])
# …identical oneshot(...) and save_pretrained(...)
```

## Colab tips & tricks

Practical things that save a session when working with models on Colab.

**Watch VRAM — it's the resource you'll run out of first.** Track allocated and peak usage; the
peak is what determines whether the next model fits.

In [ ]:
import torch

def gpu_mem(tag=""):
    alloc = torch.cuda.memory_allocated() / 1e9
    peak  = torch.cuda.max_memory_allocated() / 1e9
    print(f"{tag:22s} allocated={alloc:5.2f} GB   peak={peak:5.2f} GB")

gpu_mem("current")

**Free a model you're done with** before loading another — otherwise the old one still holds
VRAM and you OOM. `del` alone isn't enough; you need `gc.collect()` + `empty_cache()`:

```python
import gc, torch
del model                              # drop Python references first
gc.collect()
torch.cuda.empty_cache()               # return freed blocks to the driver
torch.cuda.reset_peak_memory_stats()   # so the next gpu_mem() peak is meaningful
```

If VRAM *stays* high after that, a stray reference is pinning it (a variable holding a tensor, an
output cell). The guaranteed reset is **Runtime → Restart session** — faster than hunting the leak.

**Gated / private models** (Llama, some Mistral) need a token — log in once per session:

```python
from huggingface_hub import notebook_login
notebook_login()          # paste a token from huggingface.co/settings/tokens
```

**Disk fills up too.** Every model downloads to `~/.cache/huggingface` (Colab gives ~100 GB, but a
few 7B checkpoints eat it). Check and clear:

```python
!df -h /                                    # free space
!du -sh ~/.cache/huggingface/hub/*          # what's cached
# !rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen2.5-7B-Instruct
```

**Persist your output** — Colab wipes local disk when the runtime recycles. Push the quantized
checkpoint somewhere durable *before* you close the tab: mount Drive
(`from google.colab import drive; drive.mount('/content/drive')`) and copy it there, `huggingface-cli
upload` it to a Hub repo, or `gcloud storage cp` to a bucket.

**`%pip` over `!pip`.** The `%` magic installs into the *running* kernel; a bare `!pip` can install
into a different environment and leave the import failing.

## Where to go from here

- **Persist it.** On a throwaway VM, push the checkpoint to durable storage:
  `gcloud storage cp -r ./<SAVE_DIR> gs://YOUR_BUCKET/models/` — then delete the GPU VM.
- **Serve it.** vLLM detects the quant format from the saved config, no special flags:
  `vllm serve ./<SAVE_DIR>`.
- **Prove it's good.** Compare perplexity + your custom eval against the original BF16 weights
  (chapter §5.1.3). Pick the most aggressive setting that still passes, and no further.

← Back to **[§5.1 Quantization](https://inference.kimambo.de/chapters/techniques/quantization/)**.